# US Census Income Prediction (ML Pipelines)

This notebook builds clean, reusable machine-learning pipelines to predict whether an individual's income exceeds $50K using the UCI Adult dataset. It covers:

- Data ingestion and cleaning
- Train/test splitting with stratification
- Preprocessing with `ColumnTransformer`
- Model training and evaluation
- Hyperparameter tuning
- Feature importance inspection


## Dataset

We use the [UCI Adult dataset](https://archive.ics.uci.edu/ml/datasets/adult) and fetch it directly from the public URL. The dataset includes demographic and employment-related attributes. Missing values are represented by `?` and will be handled during preprocessing.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

COLUMN_NAMES = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

raw_df = pd.read_csv(
    DATA_URL,
    header=None,
    names=COLUMN_NAMES,
    na_values="?",
    skipinitialspace=True,
)

raw_df.head()


In [ ]:
raw_df.shape


In [ ]:
raw_df.isna().sum().sort_values(ascending=False).head(10)


In [ ]:
raw_df["income"].value_counts(normalize=True)


## Feature selection

We focus on a subset of numerical and categorical features for a clean baseline pipeline.


In [ ]:
FEATURES = [
    "age",
    "education-num",
    "workclass",
    "hours-per-week",
    "sex",
    "race",
]
TARGET = "income"

X = raw_df[FEATURES].copy()
y = (raw_df[TARGET] == ">50K").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123,
    stratify=y,
)

X_train.shape, X_test.shape


## Preprocessing pipeline

- Numerical features: median imputation
- Categorical features: most-frequent imputation + one-hot encoding


In [ ]:
numeric_features = ["age", "education-num", "hours-per-week"]
categorical_features = ["workclass", "sex", "race"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)


## Model training and evaluation

We compare baseline models with a shared preprocessing pipeline.


In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "pipeline": pipeline,
    }

models = {
    "Decision Tree (stump)": DecisionTreeClassifier(max_depth=1, random_state=123),
    "AdaBoost": AdaBoostClassifier(random_state=123),
    "Gradient Boosting": GradientBoostingClassifier(random_state=123),
}

results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X_train, X_test, y_train, y_test)

pd.DataFrame({
    name: {
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    }
    for name, metrics in results.items()
}).T.sort_values(by="f1", ascending=False)


## Hyperparameter tuning (AdaBoost)

We tune the number of estimators to improve performance.


In [ ]:
ada_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", AdaBoostClassifier(random_state=123)),
    ]
)

param_grid = {
    "model__n_estimators": [25, 50, 75, 100],
    "model__learning_rate": [0.5, 1.0],
}

ada_grid = GridSearchCV(
    ada_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)

ada_grid.fit(X_train, y_train)
ada_grid.best_params_, ada_grid.best_score_


In [ ]:
best_ada = ada_grid.best_estimator_

best_preds = best_ada.predict(X_test)

print(
    classification_report(
        y_test,
        best_preds,
        target_names=['<=50K', '>50K'],
    )
)


In [ ]:
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("AdaBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Reproducibility notes

- All models set a `random_state` for repeatable runs.
- The preprocessing pipeline is encapsulated with the estimator to avoid leakage.
- For production, consider persisting the pipeline with `joblib` and monitoring drift.


## Feature importances

We inspect the most influential one-hot encoded features from the tuned AdaBoost model.


In [ ]:
def get_feature_names(preprocessor, numeric_features, categorical_features):
    numeric_names = numeric_features
    categorical_names = (
        preprocessor
        .named_transformers_["categorical"]
        .named_steps["onehot"]
        .get_feature_names_out(categorical_features)
        .tolist()
    )
    return numeric_names + categorical_names

feature_names = get_feature_names(
    best_ada.named_steps["preprocessor"],
    numeric_features,
    categorical_features,
)

importances = best_ada.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values(by="importance", ascending=False)
    .head(12)
)

feature_importance_df
